## Tema 4 - Listado 1 - Ejercicio 1

Ejercicio para entrenar un modelo simple de red neuronal utilizando la biblioteca Keras de Tensorflow.

## Carga de datos

El conjunto de datos son frases sencillas anotadas manualmente con su sentido positivo o negativo. 

* "Estoy un poco harto del día a día, nada mejora" -> Negativo
* "Hoy es un buen día" -> Positivo
* "No se te ve satisfecho con el trabajo" -> Negativo
* "Este paisaje es hermoso y bonito" -> Positivo


In [ ]:
sentences = ['Estoy un poco harto del día a día , nada mejora',
             'Hoy es un buen día',
             'No se te ve satisfecho con el trabajo',
             'Este paisaje es hermoso y bonito']

# 1: positivo, 0: negativo
labels = [0,1,0,1]


Hay que preparar los datos de entrenamiento:

* Longitud de las secuencias de texto = 10
* Tamaño del vocabulario = 50
  

In [3]:
import spacy
import spacy.cli
spacy.cli.download("es_core_news_sm")
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Cargar el modelo de español de Spacy
nlp = spacy.load('es_core_news_sm')

# Para preparar el vocabulario
def prepare_vocabulary(corpus, vocab_size):    
   
    # Crear un diccionario para mapear tokens a índices
    token_to_index = {}
    current_index = 1 # importante empezar en 1, 0 se usa para padding

    # Primera pasada para construir vocabulario
    for sentence in corpus:
        doc = nlp(sentence)
        for token in doc:
            if token.text not in token_to_index and current_index < vocab_size:
                token_to_index[token.text] = current_index
                current_index += 1
    
    return token_to_index


# representación one hot
def prepare_sentences(corpus, vocabulary, max_length):
    encoded_sentences = []
    
    # Segunda pasada para codificar oraciones
    for sentence in corpus:
        doc = nlp(sentence)
        encoded_sentence = []
        for token in doc:
            # Si el token está en nuestro vocabulario, usar su índice
            if token.text in vocabulary:
                encoded_sentence.append(vocabulary[token.text])
            # Si no está, usar el índice 0 (desconocido)
            else:
                encoded_sentence.append(0)
        encoded_sentences.append(encoded_sentence)

    # Hacer padding de las secuencias
    prepared_sentences = pad_sequences(encoded_sentences, maxlen=max_length, padding='post', truncating='post')
    print("Oraciones originales(",len(corpus),"):")
    print(corpus)  
    print("Oraciones procesadas(",len(prepared_sentences),"):")
    print(prepared_sentences)    
    return prepared_sentences

# Uso
vocab_size = 50
max_length = 10
vocabulary_train = prepare_vocabulary(sentences,vocab_size)
print("\nVocabulario (",len(vocabulary_train),"):")
print(vocabulary_train)
prepared_sentences = prepare_sentences(sentences, vocabulary_train, max_length)

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Vocabulario ( 26 ):
{'Estoy': 1, 'un': 2, 'poco': 3, 'harto': 4, 'del': 5, 'día': 6, 'a': 7, ',': 8, 'nada': 9, 'mejora': 10, 'Hoy': 11, 'es': 12, 'buen': 13, 'No': 14, 'se': 15, 'te': 16, 've': 17, 'satisfecho': 18, 'con': 19, 'el': 20, 'trabajo': 21, 'Este': 22, 'paisaje': 23, 'hermoso': 24, 'y': 25, 'bonito': 26}
Oraciones originales( 4 ):
['Estoy un poco harto del día a día , nada mejora', 'Hoy es un buen día', 'No se te ve satisfecho con el trabajo', 'Este paisaje es hermoso y bonito']
Oraciones procesadas( 4 ):
[[ 1  2  3  4  5  6  7  6  8  9]
 [11 12  2 13  6  0  0  0  0  0]
 [14 15 16 17 18 19 20 21  0  0]
 [22 23 12 24 25 26  0  0  0  0]]


## CNN

In [7]:
import tensorflow as tf
tf.__version__

'2.21.0'

## Configuramos el modelo

In [8]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Embedding, Conv1D, MaxPooling1D

# Crear una red secuencial para el modelo
model = Sequential()

# Añadir una capa inicial de embedding que transforma los índices de palabras en vectores densos
vector_size = 8
model.add(Embedding(vocab_size, vector_size))

# Añadir una capa de aplanado (Flatten) para aplanar la entrada, convirtiendo los datos multidimensionales en un vector unidimensional
model.add(Flatten())

# Añadir una capa densa con 1 neurona para una salida binaria con una función de activación sigmoid para clasificación binaria. Devuelve una probabilidad.
# Si la probabilidad es cercana a 1, la capa devuelve 1, y 0 en otro caso. 
model.add(Dense(1, activation='sigmoid'))

print("Red diseñada correctamente")


Red diseñada correctamente


## Compilamos del modelo

In [9]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model
model.build(input_shape=(None, max_length)) 
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 10, 8)          │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481 (1.88 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamos el modelo

In [10]:
from sklearn.model_selection import train_test_split
import numpy as np

# Convertir a NumPy arrays para asegurar compatibilidad y rendimiento
prepared_sentences = np.array(prepared_sentences)
labels = np.array(labels)

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(prepared_sentences, labels, test_size=0.2, random_state=42)

# Entrenar el modelo
batch_size = 32
epochs = 5
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), batch_size=batch_size, epochs=epochs)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 891ms/step - accuracy: 0.3333 - loss: 0.6906 - val_accuracy: 1.0000 - val_loss: 0.6786
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6667 - loss: 0.6856 - val_accuracy: 1.0000 - val_loss: 0.6784
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.6667 - loss: 0.6807 - val_accuracy: 1.0000 - val_loss: 0.6782
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.6667 - loss: 0.6758 - val_accuracy: 1.0000 - val_loss: 0.6780
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 1.0000 - loss: 0.6710 - val_accuracy: 1.0000 - val_loss: 0.6777


## Evaluar el modelo

Vamos a evaluarlo sobre el conjunto test. En primer lugar, vamos a obtener las métricas loss y accuracy en dicho conjunto (que no ha sido utilizado en ninguna fase del entrenamiento).


In [11]:
#loss, accuracy = model.evaluate(X_test, y_test)
#print(f"Accuracy: {accuracy * 100:.2f}%")

# Conjunto más amplio de frases de prueba
test_sentences = [
    "No fui al estreno de la película porque nadie me quería acompañar",
    "Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio",
    "Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor",
    "Al final decidí no ir al cine porque estaba cansada",
    "Todo es maravilloso y formidable, muy bonito"
]

# Preparar los datos
#prepared_test, vocabulary_test = prepare(test_sentences, vocab_size=vocab_size, max_length=max_length)
# Para recordar en la salida cuál era el vocabulario
print("\nVocabulario (",len(vocabulary_train),"):")
print(vocabulary_train)

prepared_test = prepare_sentences(test_sentences, vocabulary_train, max_length)

# Realizar predicciones
predictions = model.predict(prepared_test)

# Interpretar las predicciones con más detalle
print("Predicciones detalladas:")
for i, sentence in enumerate(test_sentences):
    pred = predictions[i][0]
    sentiment = "Positivo" if pred > 0.5 else "Negativo"
    print(f"\nTexto: {sentence}")
    print(f"Predicción numérica: {pred:.4f}")
    print(f"Sentimiento predicho: {sentiment}")


Vocabulario ( 26 ):
{'Estoy': 1, 'un': 2, 'poco': 3, 'harto': 4, 'del': 5, 'día': 6, 'a': 7, ',': 8, 'nada': 9, 'mejora': 10, 'Hoy': 11, 'es': 12, 'buen': 13, 'No': 14, 'se': 15, 'te': 16, 've': 17, 'satisfecho': 18, 'con': 19, 'el': 20, 'trabajo': 21, 'Este': 22, 'paisaje': 23, 'hermoso': 24, 'y': 25, 'bonito': 26}
Oraciones originales( 5 ):
['No fui al estreno de la película porque nadie me quería acompañar', 'Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio', 'Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor', 'Al final decidí no ir al cine porque estaba cansada', 'Todo es maravilloso y formidable, muy bonito']
Oraciones procesadas( 5 ):
[[14  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  7  0  0  0  0  0]
 [ 0  0  0  0  0  5  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0 12  0 25  0  8  0 26  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Predicciones detalladas:

T